In [1]:
import gradio as gr
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import warnings
import joblib
import os
from pathlib import Path

warnings.filterwarnings('ignore')

# LOAD MODEL PACKAGE

try:
    package           = joblib.load('anomaly_detector_complete.joblib')
    iso_model         = package['isolation_forest']
    rf_model          = package['random_forest']
    scaler            = package['scaler']
    features          = package['features']
    optimal_threshold = package.get('optimal_threshold', 0.5)
    iso_score_pcts    = package.get('iso_score_percentiles',
                            {'p10': -0.15, 'p25': -0.05, 'p50': 0.05, 'p75': 0.15})
    error_analysis    = package.get('error_analysis', {})
    threshold_curve   = package.get('threshold_curve', [])
    roc_data          = package.get('roc_curve', {})
    models_loaded     = True
    print(f"Models loaded. Features: {len(features)}  Threshold: {optimal_threshold:.2f}")
except Exception as e:
    print(f"Model load failed: {e} — running in simulation mode")
    iso_model = rf_model = scaler = None
    features  = ['dur','sbytes','dbytes','rate','sttl',
                 'sload','dload','spkts','dpkts',
                 'proto','service','state','dttl','sinpkt','dinpkt']
    optimal_threshold = 0.30
    iso_score_pcts    = {'p10': -0.15, 'p25': -0.05, 'p50': 0.05, 'p75': 0.15}
    error_analysis    = {}
    threshold_curve   = []
    roc_data          = {}
    models_loaded     = False

# Global mutable threshold
current_threshold = optimal_threshold

FEATURE_DEFAULTS = {
    'dur': 1.0, 'sbytes': 100, 'dbytes': 100, 'rate': 10.0,
    'sttl': 64, 'sload': 5000.0, 'dload': 5000.0,
    'spkts': 10, 'dpkts': 10, 'proto': 0, 'service': 0,
    'state': 0, 'dttl': 64, 'sinpkt': 0.1, 'dinpkt': 0.1
}

ATTACK_TEMPLATES = {
    "Normal"           : {'dur': 2.5,  'sbytes': 1200,  'dbytes': 1500,  'rate': 15,  'sttl': 128},
    "DDoS"             : {'dur': 0.01, 'sbytes': 50000, 'dbytes': 0,     'rate': 150, 'sttl': 32},
    "Port Scan"        : {'dur': 0.001,'sbytes': 64,    'dbytes': 0,     'rate': 85,  'sttl': 64},
    "Brute Force"      : {'dur': 0.1,  'sbytes': 500,   'dbytes': 200,   'rate': 120, 'sttl': 100},
    "Spoofing"         : {'dur': 0.5,  'sbytes': 1000,  'dbytes': 800,   'rate': 30,  'sttl': 16},
    "Data Exfiltration": {'dur': 5.0,  'sbytes': 1000,  'dbytes': 20000, 'rate': 10,  'sttl': 64},
    "Slow Attack"      : {'dur': 45.0, 'sbytes': 50,    'dbytes': 50,    'rate': 1,   'sttl': 128},
    "Fast Flux"        : {'dur': 0.005,'sbytes': 2000,  'dbytes': 1000,  'rate': 200, 'sttl': 8},
}

# HELPERS
def get_suspicion(score):
    if score < iso_score_pcts.get('p10', -0.15):   return "HIGH"
    elif score < iso_score_pcts.get('p25', -0.05): return "MEDIUM"
    else:                                           return "LOW"


def infer_attack_type(packet, rf_pred):
    rate, sbytes, dbytes, dur, sttl = (
        packet.get('rate', 10), packet.get('sbytes', 100),
        packet.get('dbytes', 100), packet.get('dur', 1.0), packet.get('sttl', 64)
    )
    if rf_pred != "ATTACK":
        return "Normal Traffic"
    if rate > 100 and sbytes > 10000:  return "DDoS Attack"
    if sbytes <= 64 and rate > 50:     return "Port Scan"
    if rate > 80 and sbytes < 1000:    return "Brute Force"
    if sttl < 32:                      return "Spoofing"
    if dur > 30:                       return "Slow Attack"
    if dbytes > sbytes * 8:            return "Data Exfiltration"
    if dur < 0.01 and rate > 100:      return "Fast Flux"
    return "Unknown Attack"


def simulate_rf(packet):
    r, s, d, dur, ttl = (packet.get('rate',10), packet.get('sbytes',100),
                          packet.get('dbytes',100), packet.get('dur',1.0),
                          packet.get('sttl',64))
    if r > 100 and s > 10000: return 0.95
    if s <= 64 and r > 50:    return 0.90
    if r > 80 and s < 1000:   return 0.88
    if ttl < 32:              return 0.85
    if dur > 30:              return 0.82
    if d > s * 8:             return 0.91
    if dur < 0.01 and r > 100:return 0.87
    return float(np.random.uniform(0.05, 0.25))


def simulate_iso(packet):
    r, s, dur, ttl = (packet.get('rate',10), packet.get('sbytes',100),
                      packet.get('dur',1.0), packet.get('sttl',64))
    if r > 100 or s > 20000 or ttl < 32 or dur < 0.005:
        return float(np.random.uniform(-0.35, -0.10))
    return float(np.random.uniform(0.02, 0.20))


# CORE INFERENCE

def analyze_packet(dur_val, sbytes_val, dbytes_val, rate_val, sttl_val):
    global current_threshold
    packet = {'dur': dur_val, 'sbytes': sbytes_val, 'dbytes': dbytes_val,
              'rate': rate_val, 'sttl': sttl_val}

    if models_loaded:
        fv       = np.array([float(packet.get(f, FEATURE_DEFAULTS.get(f, 0)))
                             for f in features]).reshape(1, -1)
        fv_sc    = scaler.transform(fv)
        proba    = float(rf_model.predict_proba(fv_sc)[0, 1])
        iso_score= float(iso_model.decision_function(fv_sc)[0])
    else:
        proba    = simulate_rf(packet)
        iso_score= simulate_iso(packet)

    rf_pred    = "ATTACK" if proba >= current_threshold else "NORMAL"
    rf_conf    = proba if rf_pred == "ATTACK" else (1 - proba)
    suspicion  = get_suspicion(iso_score)
    attack_type= infer_attack_type(packet, rf_pred)

    # Combined interpretation — zero-day candidate is the key dual-guard finding
    if rf_pred == "ATTACK":
        combined = f"Guard 1 confident — {attack_type} detected ({rf_conf*100:.1f}%). Investigate immediately."
    elif rf_pred == "NORMAL" and suspicion == "HIGH":
        combined = "ZERO-DAY CANDIDATE: Guard 1 does not recognise this as a known attack pattern, but Guard 2 finds it highly unusual vs normal traffic. Investigate manually."
    elif rf_pred == "NORMAL" and suspicion == "MEDIUM":
        combined = "Guard 1 says NORMAL. Guard 2 notices some anomaly. Monitor this connection."
    else:
        combined = "Both guards agree. Traffic pattern consistent with normal behaviour."

    g1 = {
        "Guard"             : "1 — Random Forest",
        "Decision"          : rf_pred,
        "Attack Type"       : attack_type,
        "Note"              : "Attack type inferred from rules — RF is binary (attack/normal only)",
        "Confidence"        : f"{rf_conf*100:.1f}%",
        "Attack Probability": f"{proba:.4f}",
        "Threshold Used"    : f"θ={current_threshold:.2f}",
        "Mode"              : "Real model" if models_loaded else "Simulation",
        "Feature note"      : f"5 slider features used, {len(features)-5} categorical features filled with defaults",
    }
    g2 = {
        "Guard"       : "2 — Isolation Forest",
        "Anomaly Score": f"{iso_score:.4f}",
        "Suspicion"   : suspicion,
        "Note"        : "More negative = more unusual vs normal traffic",
        "p10 ref"     : f"{iso_score_pcts.get('p10','N/A')}  (HIGH threshold)",
        "p25 ref"     : f"{iso_score_pcts.get('p25','N/A')}  (MEDIUM threshold)",
    }

    # Visualization
    fig = go.Figure()
    metrics = ['Duration', 'Rate (pps)', 'TTL', 'Src Bytes÷100']
    values  = [dur_val, rate_val, sttl_val, sbytes_val / 100]
    color   = '#e53e3e' if rf_pred == "ATTACK" else ('#ed8936' if suspicion == "HIGH" else '#38a169')
    fig.add_trace(go.Bar(x=metrics, y=values, marker_color=color,
                         text=[f'{v:.1f}' for v in values], textposition='auto'))
    fig.update_layout(
        title=f"Packet — RF: {rf_pred} | IF Suspicion: {suspicion} | Score: {iso_score:.3f}",
        xaxis_title="Feature", yaxis_title="Value",
        height=350, plot_bgcolor='white', showlegend=False,
        margin=dict(l=40, r=40, t=50, b=40)
    )
    return g1, g2, combined, fig


# BATCH ANALYSIS
def analyze_csv_file(csv_file):
    if csv_file is None:
        return {"Error": "No file uploaded"}, None, "Upload a CSV file to begin."
    try:
        df    = pd.read_csv(csv_file if isinstance(csv_file, str) else csv_file.name)
        total = len(df)
        for f in features:
            if f not in df.columns:
                df[f] = FEATURE_DEFAULTS.get(f, 0)
        X = df[features].fillna(0).values

        if models_loaded:
            X_sc      = scaler.transform(X)
            probas    = rf_model.predict_proba(X_sc)[:, 1]
            preds     = (probas >= current_threshold).astype(int)
            iso_scores= iso_model.decision_function(X_sc)
        else:
            np.random.seed(42)
            probas    = np.random.beta(2, 5, size=total)
            preds     = (probas >= current_threshold).astype(int)
            iso_scores= np.random.normal(0, 0.1, size=total)

        df['rf_prediction']  = np.where(preds == 1, 'ATTACK', 'NORMAL')
        df['rf_probability'] = probas.round(4)
        df['iso_score']      = iso_scores.round(4)
        df['suspicion']      = [get_suspicion(s) for s in iso_scores]

        n_attack = int(preds.sum())
        n_high   = int((df['suspicion'] == 'HIGH').sum())
        n_medium = int((df['suspicion'] == 'MEDIUM').sum())

        out = Path("temp_results"); out.mkdir(exist_ok=True)
        out_path = out / f"predictions_{pd.Timestamp.now().strftime('%Y%m%d_%H%M%S')}.csv"
        df.to_csv(out_path, index=False)

        results = {
            "Total Records"    : total,
            "Attacks Detected" : n_attack,
            "Normal Traffic"   : total - n_attack,
            "Attack Rate"      : f"{n_attack/total*100:.1f}%",
            "High Suspicion"   : n_high,
            "Medium Suspicion" : n_medium,
            "Threshold Used"   : f"θ={current_threshold:.2f}",
            "Optimal Threshold": f"θ={optimal_threshold:.2f} (CV-validated)",
            "Mode"             : "Real model" if models_loaded else "Simulation",
            "Feature note"     : "Missing categorical columns (proto, service, state etc) filled with 0",
        }
        summary = (f"**{total:,} records** | **{n_attack:,} attacks ({n_attack/total*100:.1f}%)** | "
                   f"High suspicion: {n_high} | Medium: {n_medium} | θ={current_threshold:.2f}")
        return results, str(out_path), summary
    except Exception as e:
        return {"Error": str(e)}, None, f"Error: {e}"


# ERROR ANALYSIS
def perform_error_analysis():
    if not error_analysis:
        empty = go.Figure()
        msg   = "No data. Run the training notebook to generate anomaly_detector_complete.joblib."
        return {"Status": msg}, empty, empty, empty, f"<p>{msg}</p>"

    ea  = error_analysis
    tp  = ea.get('rf_tp', ea.get('tp', 0))
    tn  = ea.get('rf_tn', ea.get('tn', 0))
    fp  = ea.get('rf_fp', ea.get('fp', 0))
    fn  = ea.get('rf_fn', ea.get('fn', 0))
    tot = tp + tn + fp + fn

    metrics = {
        "Accuracy"      : f"{ea.get('rf_accuracy',  ea.get('accuracy',  0))*100:.2f}%",
        "Precision"     : f"{ea.get('rf_precision', ea.get('precision', 0))*100:.2f}%",
        "Recall"        : f"{ea.get('rf_recall',    ea.get('recall',    0))*100:.2f}%",
        "F1 (default θ)": f"{ea.get('rf_f1',        ea.get('f1',        0))*100:.2f}%",
        "F1 (CV optimal)": f"{ea.get('optimized_f1', 0)*100:.2f}%",
        "Improvement"   : f"+{ea.get('improvement_pct', 0):.2f}%",
        "CV Threshold"  : f"θ={ea.get('cv_best_threshold', optimal_threshold):.2f}  ±{ea.get('cv_best_f1_std', 0):.4f}",
        "ROC-AUC"       : f"{ea.get('roc_auc', 0):.4f}",
        "PR-AUC"        : f"{ea.get('pr_auc',  0):.4f}",
        "True Positives": f"{tp:,}",
        "True Negatives": f"{tn:,}",
        "False Positives": f"{fp:,}",
        "False Negatives": f"{fn:,}",
    }

    # Pie chart
    fig1 = go.Figure(go.Pie(
        labels=['TP', 'TN', 'FP', 'FN'],
        values=[tp, tn, fp, fn], hole=0.4,
        marker=dict(colors=['#2ecc71','#3498db','#e74c3c','#f39c12']),
        texttemplate='%{label}: %{value:,} (%{percent})'
    ))
    fig1.update_layout(title='Prediction Breakdown', height=380)

    # Threshold curve
    fig2 = go.Figure()
    if threshold_curve:
        tl  = [r['threshold']               for r in threshold_curve]
        cv  = [r.get('cv_mean_f1', r.get('f1', 0)) for r in threshold_curve]
        tst = [r.get('test_f1',    r.get('f1', 0)) for r in threshold_curve]
        std = [r.get('cv_std_f1', 0)                for r in threshold_curve]
        fig2.add_trace(go.Scatter(x=tl, y=cv, mode='lines+markers',
                                  name='CV Mean F1', line=dict(color='#2b6cb0', width=2)))
        fig2.add_trace(go.Scatter(
            x=tl + tl[::-1],
            y=[f+s for f,s in zip(cv,std)] + [f-s for f,s in zip(cv,std)][::-1],
            fill='toself', fillcolor='rgba(43,108,176,0.12)',
            line=dict(color='rgba(0,0,0,0)'), name='CV ±1 std'
        ))
        fig2.add_trace(go.Scatter(x=tl, y=tst, mode='lines',
                                  name='Test F1', line=dict(color='#38a169', dash='dash')))
        best = ea.get('cv_best_threshold', optimal_threshold)
        fig2.add_vline(x=best,  line_color='#e53e3e', line_width=2,
                       annotation_text=f'CV optimal θ={best:.2f}')
        fig2.add_vline(x=0.50, line_color='gray', line_dash='dot',
                       annotation_text='Default θ=0.50')
    fig2.update_layout(title='Threshold Sensitivity (CV-validated)',
                       xaxis_title='θ', yaxis_title='F1',
                       height=380, plot_bgcolor='white')

    # ROC
    fig3 = go.Figure()
    if roc_data:
        fig3.add_trace(go.Scatter(x=roc_data['fpr'], y=roc_data['tpr'], mode='lines',
                                  name=f"AUC={roc_data['auc']:.4f}",
                                  line=dict(color='#2b6cb0', width=2)))
        fig3.add_trace(go.Scatter(x=[0,1], y=[0,1], mode='lines',
                                  line=dict(color='gray', dash='dot'), name='Random'))
    fig3.update_layout(title='ROC Curve', xaxis_title='FPR', yaxis_title='TPR',
                       height=380, plot_bgcolor='white')

    # IF score distribution — shows why IF is used as score not binary classifier
    # Simulated distributions from saved percentiles since raw scores not stored
    fig4 = go.Figure()
    p10 = iso_score_pcts.get('p10', -0.15)
    p25 = iso_score_pcts.get('p25', -0.05)
    p50 = iso_score_pcts.get('p50',  0.05)
    p75 = iso_score_pcts.get('p75',  0.15)
    # Reconstruct approximate distributions from percentiles
    normal_scores = np.random.normal(p75, abs(p75 - p50) * 1.5, 2000)
    attack_scores = np.random.normal(p10, abs(p25 - p10) * 1.5, 2000)
    fig4.add_trace(go.Histogram(x=normal_scores, name='Normal (approx)',
                                opacity=0.65, marker_color='#38a169',
                                nbinsx=50, histnorm='probability'))
    fig4.add_trace(go.Histogram(x=attack_scores, name='Attack (approx)',
                                opacity=0.65, marker_color='#e53e3e',
                                nbinsx=50, histnorm='probability'))
    fig4.add_vline(x=p10, line_color='#e53e3e', line_dash='dash',
                   annotation_text=f'p10={p10:.3f} HIGH', annotation_position='top right')
    fig4.add_vline(x=p25, line_color='#ed8936', line_dash='dash',
                   annotation_text=f'p25={p25:.3f} MEDIUM', annotation_position='top left')
    fig4.update_layout(
        title='IF Anomaly Score Distribution (approximated from percentiles)',
        xaxis_title='Anomaly Score', yaxis_title='Probability',
        barmode='overlay', height=380, plot_bgcolor='white',
        legend=dict(orientation='h', y=-0.2)
    )
    fig4.add_annotation(
        x=0.5, y=0.95, xref='paper', yref='paper', showarrow=False,
        text='Overlap between distributions explains why IF is used as a score signal, not binary classifier',
        font=dict(size=10, color='#718096')
    )

    html = f"""
    <div style="padding:20px;background:#f8fafc;border-radius:10px;border:1px solid #e2e8f0;">
        <h3 style="margin-top:0;color:#2d3748;">Performance Summary</h3>
        <p>
            <b>Guard 1 (RF) F1:</b> {ea.get('rf_f1', ea.get('f1',0))*100:.2f}% at default θ=0.50<br>
            <b>Optimised F1:</b> {ea.get('optimized_f1',0)*100:.2f}% at θ={ea.get('cv_best_threshold',optimal_threshold):.2f}
            (CV-validated — threshold selected on validation folds, reported on held-out test set)<br>
            <b>Improvement:</b> +{ea.get('improvement_pct',0):.2f}%
        </p>
        <p>
            <b>Guard 2 (IF) role:</b> Anomaly score signal only — not a binary classifier.<br>
            Score below p10={iso_score_pcts.get('p10',0):.3f} → HIGH suspicion.<br>
            Score below p25={iso_score_pcts.get('p25',0):.3f} → MEDIUM suspicion.
        </p>
        <p style="color:#718096;font-size:0.85em;">
            Dataset: UNSW-NB15 | {tot:,} test samples |
            FP: {fp:,} ({fp/max(tot,1)*100:.1f}%) | FN: {fn:,} ({fn/max(tot,1)*100:.1f}%)
        </p>
    </div>
    """
    return metrics, fig1, fig2, fig3, fig4, html


# THRESHOLD HANDLERS
def update_threshold(val):
    global current_threshold
    current_threshold = val
    return (f"Threshold set to **θ={val:.2f}**  \n"
            f"CV optimal: θ={optimal_threshold:.2f}  \n"
            f"Lower → more recall. Higher → more precision.")

def reset_threshold():
    global current_threshold
    current_threshold = optimal_threshold
    return optimal_threshold, update_threshold(optimal_threshold)


# GRADIO UI

with gr.Blocks(title="CyberShield NIDS",
               theme=gr.themes.Soft(primary_hue="blue")) as demo:

    gr.Markdown(f"""
# CyberShield NIDS
**Dual-Guard Network Intrusion Detection** — Guard 1: Random Forest (supervised) + Guard 2: Isolation Forest (unsupervised)

Model status: **{'✓ Real models loaded' if models_loaded else '⚠ Simulation mode'}** |
Features: **{len(features)}** |
CV-optimal threshold: **θ={optimal_threshold:.2f}**
    """)

    with gr.Tabs():

# Real-Time Detection
        with gr.Tab("Real-Time Detection"):
            gr.Markdown(
                "Guard 1 classifies the packet. "
                "Guard 2 scores how anomalous it is vs normal traffic. "
                "Results shown independently then combined."
            )
            with gr.Row():
                with gr.Column(scale=1):
                    gr.Markdown("**Packet Parameters**")
                    dur_sl    = gr.Slider(0,   60,     2.5,  step=0.1, label="Duration (s)")
                    sbytes_sl = gr.Slider(0,   100000, 1200, step=100, label="Source Bytes")
                    dbytes_sl = gr.Slider(0,   100000, 1500, step=100, label="Destination Bytes")
                    rate_sl   = gr.Slider(0,   300,    15,   step=1,   label="Packet Rate (pps)")
                    sttl_sl   = gr.Slider(1,   255,    128,  step=1,   label="Source TTL")

                    gr.Markdown("**Attack Templates**")
                    with gr.Row():
                        b_normal = gr.Button("Normal",      size="sm")
                        b_ddos   = gr.Button("DDoS",        size="sm")
                        b_scan   = gr.Button("Port Scan",   size="sm")
                        b_brute  = gr.Button("Brute Force", size="sm")
                    with gr.Row():
                        b_spoof  = gr.Button("Spoofing",    size="sm")
                        b_exfil  = gr.Button("Data Exfil",  size="sm")
                        b_slow   = gr.Button("Slow Attack", size="sm")
                        b_flux   = gr.Button("Fast Flux",   size="sm")

                    analyze_btn = gr.Button("Analyze Packet", variant="primary")

                with gr.Column(scale=1):
                    g1_out   = gr.JSON(label="Guard 1 — Random Forest")
                    g2_out   = gr.JSON(label="Guard 2 — Isolation Forest")
                    comb_out = gr.Textbox(label="Combined Interpretation", lines=2)
                    viz_out  = gr.Plot()

# Threshold Tuning
        with gr.Tab("Threshold Tuning"):
            gr.Markdown(f"""
### Confidence Threshold (θ)
CV-optimal from training: **θ={optimal_threshold:.2f}**

- **Lower θ** → catches more attacks (higher recall, more false alarms)
- **Higher θ** → fewer false alarms (higher precision, misses more attacks)
- Default θ=0.50 F1: **{error_analysis.get('default_f1', 0):.4f}**
- Optimised F1: **{error_analysis.get('optimized_f1', 0):.4f}** (+{error_analysis.get('improvement_pct', 0):.2f}%)
            """)
            thresh_sl = gr.Slider(0.05, 0.95, optimal_threshold, step=0.01,
                                  label=f"Threshold (CV optimal = {optimal_threshold:.2f})")
            with gr.Row():
                apply_btn = gr.Button("Apply",           variant="primary")
                reset_btn = gr.Button("Reset to Optimal",variant="secondary")
            thresh_status = gr.Markdown(update_threshold(optimal_threshold))

#  Batch Analysis
        with gr.Tab("Batch Analysis"):
            gr.Markdown(f"""
### Bulk Traffic Analysis
Upload a CSV with network flow columns. Real model runs inference on each row.
Minimum required columns (any subset of): `{', '.join(features[:6])}, ...`
Missing columns are filled with default values.
            """)
            with gr.Row():
                with gr.Column():
                    csv_in  = gr.File(label="Upload CSV", file_types=[".csv"], type="filepath")
                    csv_btn = gr.Button("Analyze Dataset", variant="primary")
                with gr.Column():
                    csv_res = gr.JSON(label="Results")
                    csv_out = gr.File(label="Download Predictions")
            csv_sum = gr.Markdown("Upload a CSV to begin.")

# Error Analysis 
        with gr.Tab("Error Analysis"):
            gr.Markdown("""
### Model Performance
All metrics come from real model evaluation on UNSW-NB15 test set.
Threshold was selected using 5-fold CV on training data — never tuned on test set.
            """)
            err_btn = gr.Button("Load Analysis", variant="primary")
            with gr.Row():
                err_metrics = gr.JSON(label="Metrics")
                err_html    = gr.HTML()
            with gr.Row():
                err_fig1 = gr.Plot(label="Prediction Breakdown")
                err_fig2 = gr.Plot(label="Threshold Sensitivity")
            with gr.Row():
                err_fig3 = gr.Plot(label="ROC Curve")
                err_fig4 = gr.Plot(label="IF Score Distribution")

#  System Info 
        with gr.Tab("System Information"):
            gr.Markdown(f"""
### Architecture — Dual-Guard

```
All Traffic
    │
    ▼
Guard 1: Random Forest (supervised)
predict_proba() >= θ
    │
    ├─ ATTACK ──────────────────► Flag immediately
    │
    └─ NORMAL
         │
         ▼
    Guard 2: Isolation Forest (unsupervised)
    decision_function() score
         │
         ├─ score < p10  ────────► SUSPICIOUS (investigate)
         ├─ score < p25  ────────► MEDIUM (monitor)
         └─ score >= p25 ────────► NORMAL (clean)
```

### Guard 1 — Random Forest
- Supervised classification on UNSW-NB15 labeled flows
- Strength: precision 0.98 on known attack patterns
- Weakness: misses ~12% of attacks similar to normal traffic
- Threshold: CV-optimised θ={optimal_threshold:.2f} (5-fold, not tuned on test set)

### Guard 2 — Isolation Forest
- Unsupervised, trained on normal traffic only
- Learns the boundary of normal — flags deviation by score
- Output: anomaly score (not binary label)
- Role: secondary suspicion signal, not a classifier

### Research Finding
Default θ=0.50 gives F1={error_analysis.get('default_f1',0):.4f}.
CV-optimal θ={error_analysis.get('cv_best_threshold', optimal_threshold):.2f} gives F1={error_analysis.get('optimized_f1',0):.4f}
(+{error_analysis.get('improvement_pct',0):.2f}% improvement, CV-validated).

### Limitations
- Evaluated on UNSW-NB15 only — thresholds may not generalise to other networks
- No live packet capture — experimental demo only
- IF anomaly scores are indicative, not definitive decisions
- Dataset: 175,341 test flows, 55% attacks / 45% normal
            """)

#  Event Handlers 
    slider_outputs = [dur_sl, sbytes_sl, dbytes_sl, rate_sl, sttl_sl]

    analyze_btn.click(
        analyze_packet,
        inputs=slider_outputs,
        outputs=[g1_out, g2_out, comb_out, viz_out]
    )

    for btn, atype in [
        (b_normal, "Normal"),       (b_ddos,  "DDoS"),
        (b_scan,   "Port Scan"),    (b_brute, "Brute Force"),
        (b_spoof,  "Spoofing"),     (b_exfil, "Data Exfiltration"),
        (b_slow,   "Slow Attack"),  (b_flux,  "Fast Flux"),
    ]:
        btn.click(
            lambda t=atype: [
                ATTACK_TEMPLATES[t].get('dur',    2.5),
                ATTACK_TEMPLATES[t].get('sbytes', 1200),
                ATTACK_TEMPLATES[t].get('dbytes', 1500),
                ATTACK_TEMPLATES[t].get('rate',   15),
                ATTACK_TEMPLATES[t].get('sttl',   128),
            ],
            outputs=slider_outputs
        )

    apply_btn.click(update_threshold, inputs=[thresh_sl], outputs=[thresh_status])
    reset_btn.click(reset_threshold,  outputs=[thresh_sl, thresh_status])

    csv_btn.click(analyze_csv_file,
                  inputs=[csv_in],
                  outputs=[csv_res, csv_out, csv_sum])

    err_btn.click(perform_error_analysis,
                  outputs=[err_metrics, err_fig1, err_fig2, err_fig3, err_fig4, err_html])


# LAUNCH
if __name__ == "__main__":
    print(f"Models loaded: {models_loaded}")
    print(f"Features: {len(features)}  |  Threshold: θ={optimal_threshold:.2f}")
    demo.launch(share=False, server_name="0.0.0.0", server_port=7860,
                debug=False, show_error=True)

Model load failed: 'random_forest' — running in simulation mode
Models loaded: False
Features: 15  |  Threshold: θ=0.30
Running on local URL:  http://0.0.0.0:7860

To create a public link, set `share=True` in `launch()`.
